**Testime roll B funktsioone päris- ja näidisandmetega**

In [ ]:
# Impordime vajalikud teegid
from supabase import create_client
from dotenv import load_dotenv
import os
import pandas as pd

# Laeme Supabase'i ühendusandmed
load_dotenv()

# Loome Supabase'i ühenduse
supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

# Funktsioon kõigi ridade laadimiseks Supabase'ist
def get_all_rows(table_name):
    all_data = []
    start = 0
    batch_size = 1000

    while True:
        response = (
            supabase.table(table_name)
            .select("*")
            .range(start, start + batch_size - 1)
            .execute()
        )

        data = response.data
        all_data.extend(data)

        if len(data) < batch_size:
            break

        start += batch_size

    return pd.DataFrame(all_data)

# Laeme müügi- ja kliendiandmed
df_orders = get_all_rows("sales")
df_customers = get_all_rows("customers")

print(f"Müügiandmeid: {len(df_orders)}, Kliente: {len(df_customers)}")

In [ ]:
# Impordime transform.py failist neli andmete töötlemise funktsiooni
from transform import (
    clean_data,
    calculate_weekly_aggregates,
    calculate_kpis,
    merge_datasets
)

**Test 1 - andmete puhastamine**

In [ ]:
# Puhastame müügiandmed, kasutades clean_data() funktsiooni
df_clean = clean_data(df_orders)

# Võrdleme ridade arvu enne ja pärast puhastamist
print("Ridu enne puhastamist:", len(df_orders))
print("Ridu pärast puhastamist:", len(df_clean))

# Kontrollime, kas puhastatud andmetes esineb duplikaate
print("Duplikaate pärast puhastamist:",
      df_clean.duplicated().sum())

# Kontrollime, kas müügikuupäevades on puuduvaid väärtusi
print("Puuduvaid kuupäevi:",
      df_clean['sale_date'].isna().sum())

# Kontrollime, kas müügisummades on puuduvaid väärtusi
print("Puuduvaid hindu:",
      df_clean['total_price'].isna().sum())

# Kuvame puhastatud andmete esimesed viis rida
display(df_clean.head())

# Automaatsed kontrollid (assert)
# Veendume, et duplikaadid ja puuduvad väärtused on eemaldatud
assert df_clean.duplicated().sum() == 0
assert df_clean['sale_date'].isna().sum() == 0
assert df_clean['total_price'].isna().sum() == 0

# Kui kõik kontrollid õnnestuvad, kuvame kinnituse
print("Andmete puhastamise kontroll edukalt läbitud!")

Testime andmete puhastamise funktsiooni ka näidisandmetega, kuhu lisame teadlikult duplikaadi ja puuduva kuupäeva. Kuna pärisandmetes neid probleeme ei esine, võimaldab näidisandmestik kontrollida, kas funktsioon eemaldab vigased read õigesti.

In [ ]:
# Loome näidisandmed, mis sisaldavad duplikaati ja puuduvat kuupäeva
df_test = pd.DataFrame({
    'sale_date': ['2024-01-01', '2024-01-01', None],
    'total_price': [100, 100, 200],
    'customer_id': [1, 1, 2]
})

# Puhastame näidisandmed clean_data() funktsiooniga
df_test_clean = clean_data(df_test)

# Kontrollime, et alles jäi üks korrektne rida
assert len(df_test_clean) == 1

# Kontrollime, et puuduvad kuupäevad on eemaldatud
assert df_test_clean['sale_date'].isna().sum() == 0

# Kui kõik kontrollid õnnestuvad, kuvame kinnituse
print("Andmete puhastamise test edukalt läbitud!")

**Test 2 - nädalapõhised kokkuvõtted**

In [ ]:
# Arvutame puhastatud müügiandmete põhjal nädalapõhised kokkuvõtted
df_weekly = calculate_weekly_aggregates(df_clean)

# Kuvame nädalapõhise kokkuvõttetabeli esimesed 10 rida
display(df_weekly.head(10))

# Kuvame nädalate arvu
print("Nädalate arv:", len(df_weekly))

# Kuvame kõigi nädalate kogukäibe
print("Kogukäive:", round(df_weekly['total_revenue'].sum(), 2), "EUR")

# Kontrollime, et nädalakokkuvõttes on vajalikud veerud
assert 'sale_date' in df_weekly.columns
assert 'total_revenue' in df_weekly.columns
assert 'order_count' in df_weekly.columns
assert 'avg_order_value' in df_weekly.columns

# Kontrollime, et nädalate kogukäive vastab algandmete kogukäibele
assert abs(
    df_weekly['total_revenue'].sum() - df_clean['total_price'].sum()
) < 0.01

# Kontrollime, et nädalate müügiridade koguarv vastab algandmetele
assert df_weekly['order_count'].sum() == len(df_clean)

# Kui kõik kontrollid õnnestuvad, kuvame kinnituse
print("Nädalapõhise kokkuvõtte kontroll edukalt läbitud!")

Testime nädalapõhiseid kokkuvõtteid näidisandmetega, et kontrollida, kas funktsioon arvutab õigesti. Näidisandmete puhul teame tulemusi ette.

In [ ]:
# Loome näidisandmed kolme tellimusega kahest erinevast nädalast
df_test_weekly = pd.DataFrame({
    'sale_date': pd.to_datetime([
        '2024-01-01',
        '2024-01-02',
        '2024-01-08'
    ]),
    'total_price': [100, 200, 400]
})

# Arvutame näidisandmete põhjal nädalapõhised kokkuvõtted
df_weekly_test = calculate_weekly_aggregates(df_test_weekly)

# Kuvame nädalapõhise kokkuvõttetabeli
display(df_weekly_test)

# Kontrollime, et andmed on koondatud kaheks nädalaks
assert len(df_weekly_test) == 2

# Kontrollime, et kogukäive on 700 eurot
assert df_weekly_test['total_revenue'].sum() == 700

# Kontrollime, et tellimuste koguarv on 3
assert df_weekly_test['order_count'].sum() == 3

# Kontrollime mõlema nädala keskmise tellimuse väärtust
assert df_weekly_test['avg_order_value'].iloc[0] == 150
assert df_weekly_test['avg_order_value'].iloc[1] == 400

# Kui kõik kontrollid õnnestuvad, kuvame kinnituse
print("Nädalapõhise kokkuvõtte test edukalt läbitud!")

**Test 3 - KPI-d**

In [ ]:
# Arvutame puhastatud müügiandmete põhjal KPI-d
kpis = calculate_kpis(df_clean)

# Kuvame arvutatud KPI-de väärtused
print(kpis)

# Kontrollime, et tulemuseks on sõnastik (dict)
assert isinstance(kpis, dict)

# Kontrollime, et sõnastikus on vähemalt kolm KPI-d
assert len(kpis) >= 3

# Kontrollime, et kogukäive on positiivne
assert kpis['total_revenue'] > 0

# Kontrollime, et unikaalsete klientide arv on positiivne
assert kpis['unique_customers'] > 0

# Kontrollime, et keskmine tellimuse väärtus on positiivne
assert kpis['avg_order_value'] > 0

# Kui kõik kontrollid õnnestuvad, kuvame kinnituse
print("KPI-de test edukalt läbitud!")

Testime KPI-sid näidisandmetega, et kontrollida, kas calculate_kpis() arvutab õigesti kolm nõutud näitajat, kasutades andmeid, mille õiged tulemused on meile ette teada.

In [ ]:
# Loome näidisandmed kolme müügirea ja kahe unikaalse kliendiga
df_test_kpis = pd.DataFrame({
    'customer_id': [1, 1, 2],
    'total_price': [100, 200, 300]
})

# Arvutame näidisandmete põhjal KPI-d
kpis_test = calculate_kpis(df_test_kpis)

# Kuvame arvutatud KPI-de väärtused
print(kpis_test)

# Kontrollime, et kogukäive on 600 eurot
assert kpis_test['total_revenue'] == 600

# Kontrollime, et unikaalseid kliente on 2
assert kpis_test['unique_customers'] == 2

# Kontrollime, et keskmine müügirea väärtus on 200 eurot
assert kpis_test['avg_order_value'] == 200

# Kui kõik kontrollid õnnestuvad, kuvame kinnituse
print("KPI-de test edukalt läbitud!")

**Test 4 - tabelite ühendamine**

In [ ]:
# Ühendame puhastatud müügiandmed kliendiandmetega
df_merged = merge_datasets(df_clean, df_customers)

# Võrdleme ridade arvu enne ja pärast ühendamist
print("Müügiridu enne ühendamist:", len(df_clean))
print("Ridu pärast ühendamist:", len(df_merged))

# Kuvame ühendatud tabeli esimesed viis rida
display(df_merged.head())

# Kontrollime, et ühendamisel jäi ridade arv samaks
assert len(df_merged) == len(df_clean)

# Kontrollime, et kliendi ees- ja perekonnanime veerud on olemas
assert 'first_name' in df_merged.columns
assert 'last_name' in df_merged.columns

# Kontrollime, et kliendi ID 2588 juurde kuulub õige eesnimi
assert (
    df_merged.loc[
        df_merged['customer_id'] == 2588,
        'first_name'
    ].iloc[0] == 'Hille'
)

# Kui kõik kontrollid õnnestuvad, kuvame kinnituse
print("Tabelite ühendamise test edukalt läbitud!")

**Test 5 - Testime veakäsitlust**

In [ ]:
# Loome vigase näidistabeli, kus puudub total_price veerg
df_vigane = pd.DataFrame({
    'customer_id': [1, 2, 3]
})

# Kontrollime, kas funktsioon annab arusaadava veateate
try:
    calculate_kpis(df_vigane)
except KeyError:
    print("Veakäsitluse test edukalt läbitud!")